# Baseline 1: The Discrete Transformer (Predictive Collapse)
**The Paradigm:** Standard autoregressive sequence modeling.

**The Goal:** Train a discrete-time Transformer on the first 50 hours of the 114-D CHRONOS Tensor (Homeostasis & Pre-Crash Wobble). Predict the system's continuous trajectory from $T=50$ hours to $T=72$ hours to identify the exact Waddington Bifurcation Point (The Crash).

*Hypothesis:* The Transformer will fail to capture the continuous-time dynamics due to the 5-minute sparsity mask and the non-Markovian phase transition. It will minimize Mean Squared Error (MSE) by predicting the mean, entirely missing the catastrophic biological variance spike.

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[*] Booting sequence model on: {device}")

In [ ]:
# --- 1. INGEST & NAIVE PREPROCESSING ---
print("[*] Loading CHRONOS 114-D Tensor...")
data = np.load('../dataset/chronos_oracle_level1.npz')

phase_100d = data['phase_100d']       # (4320, 100)
telemetry_14d = data['telemetry_14d'] # (4320, 14) with NaNs
true_crash_step = data['ground_truth_crash'][0]

print(f"[*] Total Time Steps: {phase_100d.shape[0]} (72 Hours at 1-min resolution)")
print(f"[*] Ground Truth Crash Step: {true_crash_step} (Hour {true_crash_step/60:.2f})")

# THE ML BRO HACK: Transformers can't handle NaNs or irregular time natively.
# Standard operating procedure: Forward fill the chemical telemetry.
# (This completely destroys the biological kinetic rates, but it forces the code to compile).
df_telemetry = pd.DataFrame(telemetry_14d)
telemetry_imputed = df_telemetry.ffill().fillna(0).values

# Stack to create the 114-D input
full_tensor = np.concatenate([phase_100d, telemetry_imputed], axis=1) # (4320, 114)

# Split Train (First 50 Hours) / Test (Hour 50 to 72)
TRAIN_STEPS = 3000
train_data = full_tensor[:TRAIN_STEPS]

# Normalize based on homeostasis (train) data
mean = train_data.mean(axis=0)
std = train_data.std(axis=0) + 1e-8
train_data_scaled = (train_data - mean) / std
full_tensor_scaled = (full_tensor - mean) / std

In [ ]:
# --- 2. DATASET & ARCHITECTURE ---
def create_sequences(data, seq_length=60):
    xs, ys = [], []
    # Predict the next immediate step based on the past hour (60 mins)
    for i in range(len(data) - seq_length):
        xs.append(data[i:(i + seq_length)])
        ys.append(data[i + seq_length])
    return torch.tensor(np.array(xs), dtype=torch.float32), torch.tensor(np.array(ys), dtype=torch.float32)

SEQ_LEN = 60 # 1 hour lookback
X_train, y_train = create_sequences(train_data_scaled, SEQ_LEN)
X_train, y_train = X_train.to(device), y_train.to(device)

class DiscreteBiologicalTransformer(nn.Module):
    def __init__(self, input_dim=114, d_model=128, nhead=8, num_layers=3):
        super(DiscreteBiologicalTransformer, self).__init__()
        # Project 114-D input to 128-D for multi-head attention division
        self.embedding = nn.Linear(input_dim, d_model)
        
        # Basic positional encoding
        self.pos_encoder = nn.Parameter(torch.randn(1, SEQ_LEN, d_model) * 0.02)
        
        self.encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=256, batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(self.encoder_layer, num_layers=num_layers)
        
        # Project back to 114-D biological space
        self.decoder = nn.Linear(d_model, input_dim)
        
    def forward(self, x):
        # x shape: [batch, seq_len, features]
        x = self.embedding(x) + self.pos_encoder
        out = self.transformer_encoder(x)
        # We only care about predicting the next step, so we decode the final token's context
        out = self.decoder(out[:, -1, :]) 
        return out

model = DiscreteBiologicalTransformer().to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
# --- 3. TRAINING LOOP (THE ILLUSION OF SUCCESS) ---
print("[*] Training discrete sequence model on T=0 to T=50 Hours...")
EPOCHS = 30
BATCH_SIZE = 128

model.train()
for epoch in range(EPOCHS):
    permutation = torch.randperm(X_train.size()[0])
    epoch_loss = 0
    for i in range(0, X_train.size()[0], BATCH_SIZE):
        indices = permutation[i:i+BATCH_SIZE]
        batch_x, batch_y = X_train[indices], y_train[indices]
        
        optimizer.zero_grad()
        output = model(batch_x)
        loss = criterion(output, batch_y)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item() * len(indices)
        
    if (epoch+1) % 5 == 0:
        print(f"Epoch {epoch+1:02d}/{EPOCHS} | MSE Loss: {epoch_loss/len(X_train):.6f}")

print("[+] Training converged. The model believes it has mapped the organoid.")

In [ ]:
# --- 4. AUTOREGRESSIVE ROLLOUT (THE REALITY CHECK) ---
print("\n[*] Executing Autoregressive Rollout: T=50 to T=72 Hours...")
model.eval()

rollout_predictions = []
# Seed the rollout with the last 60 minutes of the training data
current_seq = X_train[-1].unsqueeze(0).clone() # Shape: [1, 60, 114]

# Roll out for the remaining 1320 steps (Hour 50 to 72) completely blind
with torch.no_grad():
    for _ in range(full_tensor.shape[0] - TRAIN_STEPS):
        next_step = model(current_seq) # Predict next minute
        rollout_predictions.append(next_step.cpu().numpy()[0])
        
        # Append new prediction and slide window forward
        next_step_unsqueezed = next_step.unsqueeze(1)
        current_seq = torch.cat((current_seq[:, 1:, :], next_step_unsqueezed), dim=1)

rollout_predictions = np.array(rollout_predictions)

# Un-normalize back to biological units
rollout_predictions = (rollout_predictions * std) + mean

In [ ]:
# --- 5. THE DAB METRIC VISUALIZATION (THE TRAP SPRINGS) ---
# In continuous biological dynamics, structural collapse (apoptosis) is preceded 
# by "Critical Slowing Down"—a massive explosion in spatial variance. 
# We calculate the variance of the 100-D Phase tensor across its spatial dimensions.

true_test_phase = full_tensor[TRAIN_STEPS:, :100]
pred_test_phase = rollout_predictions[:, :100]

true_variance = np.var(true_test_phase, axis=1)
pred_variance = np.var(pred_test_phase, axis=1)
time_axis = np.arange(TRAIN_STEPS, full_tensor.shape[0]) / 60.0 # In Hours

plt.figure(figsize=(12, 6))
plt.style.use('dark_background')

plt.plot(time_axis, true_variance, label="Ground Truth Biology (LLSM QPI)", color="crimson", linewidth=2)
plt.plot(time_axis, pred_variance, label="Transformer Rollout", color="dodgerblue", linestyle="--", linewidth=2)

plt.axvline(x=true_crash_step/60.0, color='white', linestyle=':', linewidth=2, label=f"Waddington Bifurcation (Crash at {true_crash_step/60.0:.1f}h)")

plt.title("Directional Attractor Basin (DAB): Structural Phase Variance vs Time", fontsize=14, fontweight='bold', color='white')
plt.xlabel("Time (Hours)", fontsize=12, color='white')
plt.ylabel("100-D Structural Phase Variance", fontsize=12, color='white')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

### The Post-Mortem Assessment
If you look at the training loop, the Transformer successfully minimized the Mean Squared Error (MSE). According to standard AI metrics, this model successfully "converged."

**But look at the graph.** 
At exactly 61.4 hours, the biological system crossed a Waddington bifurcation point. The active-matter physics of the organelle network lost its elasticity, resulting in a catastrophic variance explosion (apoptosis / structural shattering). 

**The Transformer completely missed it.** It decayed to the mean. It predicted homeostasis. 
Why?
1. **Discrete vs Continuous Time:** By using `.fillna('ffill')`, we destroyed the non-Markovian memory of the sparse Lanthanide chemical flashes. We forced continuous physics into discrete sequence tokens.
2. **Mean-Seeking Loss:** MSE penalizes extreme variance. It mathematically punishes the model for predicting a crash. 

If this were a real drug-screening pipeline for an FDA trial, this AI would have predicted the tissue survived perfectly. The patient dies.